In [1]:
# Adaptive Search Agent – AUTO OUTPUT (NO INPUT)
# Prints ALL 50 scenarios + saves output to file
# Run in VS Code (Python 3)

import heapq
import random
import time
import sys

OUTPUT_FILE = "adaptive_output.txt"

# ---------- A* SEARCH ----------
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    open_list = []
    heapq.heappush(open_list, (manhattan(start, goal), 0, start))
    came_from = {}
    g_cost = {start: 0}

    while open_list:
        _, cost, current = heapq.heappop(open_list)

        if current == goal:
            path = []
            while current != start:
                path.append(current)
                current = came_from[current]
            return path[::-1]

        for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
            nx, ny = current[0] + dx, current[1] + dy
            if 0 <= nx < rows and 0 <= ny < cols and grid[nx][ny] == 0:
                new_cost = cost + 1
                if (nx, ny) not in g_cost or new_cost < g_cost[(nx, ny)]:
                    g_cost[(nx, ny)] = new_cost
                    priority = new_cost + manhattan((nx, ny), goal)
                    heapq.heappush(open_list, (priority, new_cost, (nx, ny)))
                    came_from[(nx, ny)] = current
    return None

# ---------- DYNAMIC ENVIRONMENT ----------
def update_environment(grid, prob=0.05):
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if random.random() < prob:
                grid[i][j] = 1 - grid[i][j]

# ---------- PRINT GRID ----------
def format_grid(grid, start, goal):
    lines = []
    for i in range(len(grid)):
        row = []
        for j in range(len(grid[0])):
            if (i, j) == start:
                row.append("A")
            elif (i, j) == goal:
                row.append("G")
            elif grid[i][j] == 1:
                row.append("#")
            else:
                row.append(".")
        lines.append(" ".join(row))
    return "\n".join(lines)

# ---------- AGENT ----------
def adaptive_agent(grid, start, goal):
    current = start
    path = astar(grid, current, goal)
    if path is None:
        return False, 0, 0

    path_len = 0
    replanning_time = 0

    while current != goal:
        update_environment(grid)

        if not path or grid[path[0][0]][path[0][1]] == 1:
            t0 = time.time()
            path = astar(grid, current, goal)
            replanning_time += time.time() - t0
            if path is None:
                return False, path_len, replanning_time

        current = path.pop(0)
        path_len += 1

    return True, path_len, replanning_time

# ---------- MAIN ----------
def run():
    scenarios = 50
    size = 8

    with open(OUTPUT_FILE, "w") as f:
        for s in range(1, scenarios + 1):
            header = f"\n{'='*50}\nSCENARIO {s} / {scenarios}\n{'='*50}\n"
            print(header)
            f.write(header)

            grid = [[0 if random.random() > 0.25 else 1 for _ in range(size)]
                    for _ in range(size)]

            start = (0, 0)
            goal = (size - 1, size - 1)
            grid[0][0] = 0
            grid[goal[0]][goal[1]] = 0

            init_grid = "\nInitial Grid:\n" + format_grid(grid, start, goal) + "\n"
            print(init_grid)
            f.write(init_grid)

            success, path_len, replan_time = adaptive_agent(grid, start, goal)

            final_grid = "\nFinal Grid:\n" + format_grid(grid, start, goal) + "\n"
            print(final_grid)
            f.write(final_grid)

            result = (
                f"Success: {success}\n"
                f"Path Length: {path_len}\n"
                f"Replanning Time: {round(replan_time,4)} seconds\n"
            )
            print(result)
            f.write(result)

    print("\nALL 50 SCENARIOS PRINTED AND SAVED TO adaptive_output.txt")

# ---------- RUN ----------
if __name__ == "__main__":
    run()


SCENARIO 1 / 50


Initial Grid:
A . # . # . # .
# # # # . . . .
. . . . . . . #
. # # . . . # .
. . # # . . # #
. . . . # # . .
. # . . . . . #
# # . . . . . G


Final Grid:
A . # . # . # .
# # # # . . . .
. . . . . . . #
. # # . . . # .
. . # # . . # #
. . . . # # . .
. # . . . . . #
# # . . . . . G

Success: False
Path Length: 0
Replanning Time: 0 seconds


SCENARIO 2 / 50


Initial Grid:
A . . . # # . .
# . . . . . . .
. . . . . . . .
# # . . . . . #
. # . . . . . .
. . . . . . . .
. # . # . . . .
. . . # . . . G


Final Grid:
A . # . # # . .
# . . # # . # .
. . # . . # # .
. . . # . . . #
# # . # . # . #
# . . # . # # #
. . . # # . . #
. # . . . . . G

Success: False
Path Length: 10
Replanning Time: 0.0003 seconds


SCENARIO 3 / 50


Initial Grid:
A . . . . # # .
. . # . # # # .
. . . . . . . .
# # . . # . # #
. # # . . . # .
. # . . . . . .
. . # # # . . .
. . # . . . . G


Final Grid:
A . . . . # # #
. . # . # # . .
# # . # . . # #
. . . . # # # #
. # # . # . # .
. # . . . . . #

In [2]:
# cyber_expert_system.py
# Mini Expert System for Cybersecurity Threat Diagnosis
# Implements forward chaining, user input, and testing on 10 scenarios

class CyberExpertSystem:
    def __init__(self):
        self.facts = {}
        self.trace = []

    # Ask user for input if fact is missing
    def ask_user(self, question):
        answer = input(question + " (yes/no): ").strip().lower()
        return answer == "yes"

    # Collect facts from user
    def get_facts(self):
        self.facts["high_traffic"] = self.ask_user("Is there unusually high network traffic?")
        self.facts["no_user_activity"] = self.ask_user("Is there no legitimate user activity?")
        self.facts["multiple_ports"] = self.ask_user("Are multiple ports being accessed rapidly?")
        self.facts["unknown_exe"] = self.ask_user("Is an unknown executable detected?")

    # Forward chaining inference
    def infer(self):
        if self.facts["high_traffic"] and self.facts["no_user_activity"]:
            self.trace.append("Rule fired: IF high traffic AND no user activity THEN DDoS")
            return "DDoS Attack"

        if self.facts["multiple_ports"] and self.facts["no_user_activity"]:
            self.trace.append("Rule fired: IF multiple ports accessed AND no user activity THEN Port Scan")
            return "Port Scanning"

        if self.facts["unknown_exe"]:
            self.trace.append("Rule fired: IF unknown executable detected THEN Malware")
            return "Malware Infection"

        self.trace.append("No threat rules matched → Normal Traffic")
        return "Normal Traffic"

    # Diagnose based on collected facts
    def diagnose(self):
        self.get_facts()
        threat = self.infer()
        return threat

# ------------------------
# Testing on 10 scenarios
# ------------------------

def test_scenarios():
    scenarios = [
        {"high_traffic": True, "no_user_activity": True, "multiple_ports": False, "unknown_exe": False, "expected": "DDoS Attack"},
        {"high_traffic": False, "no_user_activity": True, "multiple_ports": True, "unknown_exe": False, "expected": "Port Scanning"},
        {"high_traffic": False, "no_user_activity": False, "multiple_ports": False, "unknown_exe": True, "expected": "Malware Infection"},
        {"high_traffic": False, "no_user_activity": False, "multiple_ports": False, "unknown_exe": False, "expected": "Normal Traffic"},
        {"high_traffic": True, "no_user_activity": True, "multiple_ports": True, "unknown_exe": False, "expected": "DDoS Attack"},
        {"high_traffic": False, "no_user_activity": False, "multiple_ports": False, "unknown_exe": True, "expected": "Malware Infection"},
        {"high_traffic": False, "no_user_activity": True, "multiple_ports": True, "unknown_exe": False, "expected": "Port Scanning"},
        {"high_traffic": False, "no_user_activity": False, "multiple_ports": False, "unknown_exe": False, "expected": "Normal Traffic"},
        {"high_traffic": True, "no_user_activity": True, "multiple_ports": False, "unknown_exe": False, "expected": "DDoS Attack"},
        {"high_traffic": False, "no_user_activity": False, "multiple_ports": False, "unknown_exe": True, "expected": "Malware Infection"},
    ]

    correct_count = 0

    for i, s in enumerate(scenarios, 1):
        system = CyberExpertSystem()
        system.facts = {
            "high_traffic": s["high_traffic"],
            "no_user_activity": s["no_user_activity"],
            "multiple_ports": s["multiple_ports"],
            "unknown_exe": s["unknown_exe"]
        }
        result = system.infer()
        print(f"Scenario {i}: Expected = {s['expected']}, Diagnosed = {result}")
        print("Reasoning Trace:", system.trace)
        print("-" * 50)
        if result == s["expected"]:
            correct_count += 1

    accuracy = (correct_count / len(scenarios)) * 100
    print(f"Accuracy on 10 test scenarios: {accuracy}%")

# ------------------------
# Main Program
# ------------------------

def main():
    print("Cybersecurity Threat Diagnosis Expert System")
    print("Do you want to run test scenarios automatically? (yes/no)")
    choice = input().strip().lower()
    if choice == "yes":
        test_scenarios()
    else:
        system = CyberExpertSystem()
        threat = system.diagnose()
        print("\nDiagnosis Result:", threat)
        print("Reasoning Trace:", system.trace)

if __name__ == "__main__":
    main()

Cybersecurity Threat Diagnosis Expert System
Do you want to run test scenarios automatically? (yes/no)
Scenario 1: Expected = DDoS Attack, Diagnosed = DDoS Attack
Reasoning Trace: ['Rule fired: IF high traffic AND no user activity THEN DDoS']
--------------------------------------------------
Scenario 2: Expected = Port Scanning, Diagnosed = Port Scanning
Reasoning Trace: ['Rule fired: IF multiple ports accessed AND no user activity THEN Port Scan']
--------------------------------------------------
Scenario 3: Expected = Malware Infection, Diagnosed = Malware Infection
Reasoning Trace: ['Rule fired: IF unknown executable detected THEN Malware']
--------------------------------------------------
Scenario 4: Expected = Normal Traffic, Diagnosed = Normal Traffic
Reasoning Trace: ['No threat rules matched → Normal Traffic']
--------------------------------------------------
Scenario 5: Expected = DDoS Attack, Diagnosed = DDoS Attack
Reasoning Trace: ['Rule fired: IF high traffic AND no us

In [4]:
import random

# --- Tic-Tac-Toe Game State ---
class TicTacToeState:
    def __init__(self, board=None, player='X'):
        self.board = board or [' '] * 9
        self.player = player  # 'X' or 'O'

    def is_terminal(self):
        return self.check_winner() is not None or ' ' not in self.board

    def get_legal_moves(self):
        moves = []
        for i in range(9):
            if self.board[i] == ' ':
                new_board = self.board.copy()
                new_board[i] = self.player
                moves.append(TicTacToeState(new_board, 'O' if self.player == 'X' else 'X'))
        return moves

    def evaluate(self):
        winner = self.check_winner()
        if winner == 'X':
            return 1
        elif winner == 'O':
            return -1
        else:
            return 0

    def check_winner(self):
        lines = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]
        for a,b,c in lines:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]
        return None

    def print_board(self):
        print(f"{self.board[0]}|{self.board[1]}|{self.board[2]}")
        print("-+-+-")
        print(f"{self.board[3]}|{self.board[4]}|{self.board[5]}")
        print("-+-+-")
        print(f"{self.board[6]}|{self.board[7]}|{self.board[8]}")
        print()


# --- Minimax with Alpha-Beta Pruning and State Printing ---
states_visited = 0  # global counter

def minimax_print(state, depth, alpha, beta, maximizing_player, verbose=True):
    global states_visited
    states_visited += 1

    if verbose:
        print(f"Exploring state (Player {state.player}, Depth {depth}):")
        state.print_board()

    if depth == 0 or state.is_terminal():
        return state.evaluate()

    if maximizing_player:
        max_eval = -float('inf')
        for child in state.get_legal_moves():
            eval = minimax_print(child, depth-1, alpha, beta, False, verbose)
            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = float('inf')
        for child in state.get_legal_moves():
            eval = minimax_print(child, depth-1, alpha, beta, True, verbose)
            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval


def get_best_move(state, depth, verbose=True):
    best_score = -float('inf') if state.player == 'X' else float('inf')
    best_move = None

    for child in state.get_legal_moves():
        score = minimax_print(child, depth-1, -float('inf'), float('inf'), state.player == 'O', verbose)
        if state.player == 'X' and score > best_score:
            best_score = score
            best_move = child
        elif state.player == 'O' and score < best_score:
            best_score = score
            best_move = child

    return best_move


# --- Single Game ---
def play_single_game(verbose=True):
    global states_visited
    states_visited = 0
    state = TicTacToeState()
    turn = 0

    while not state.is_terminal():
        if verbose:
            print(f"\nTurn {turn + 1}: Player {state.player}'s move\n")
        if state.player == 'X':
            state = get_best_move(state, depth=3, verbose=verbose)
        else:
            moves = state.get_legal_moves()
            state = random.choice(moves)
        turn += 1

        if verbose:
            print(f"Board after move:\n")
            state.print_board()

    winner = state.check_winner()
    if verbose:
        print("Final Board:")
        state.print_board()
        if winner:
            print(f"Winner: {winner}")
        else:
            print("Draw!")
        print(f"Total states visited in this game: {states_visited}\n")

    # Return game result: 1 if X wins, -1 if O wins, 0 if draw
    if winner == 'X':
        return 1
    elif winner == 'O':
        return -1
    else:
        return 0


# --- Play Multiple Games ---
def play_multiple_games(n_games=5, verbose=False):
    x_wins = 0
    o_wins = 0
    draws = 0
    total_score = 0

    for i in range(n_games):
        if verbose:
            print(f"\n=== Game {i+1} ===")
        result = play_single_game(verbose=verbose)
        total_score += result
        if result == 1:
            x_wins += 1
        elif result == -1:
            o_wins += 1
        else:
            draws += 1

    print("\n--- Summary ---")
    print(f"Total games played: {n_games}")
    print(f"X wins: {x_wins}")
    print(f"O wins: {o_wins}")
    print(f"Draws: {draws}")
    print(f"Total score (X vs O): {total_score}")


# --- Run Multiple Games ---
if __name__ == "__main__":
    # Set verbose=True to print every state explored during Minimax
    play_multiple_games(n_games=2, verbose=True)


=== Game 1 ===

Turn 1: Player X's move

Exploring state (Player O, Depth 2):
X| | 
-+-+-
 | | 
-+-+-
 | | 

Exploring state (Player X, Depth 1):
X|O| 
-+-+-
 | | 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|O|X
-+-+-
 | | 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
X| | 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
 |X| 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
 | |X
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
 | | 
-+-+-
X| | 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
 | | 
-+-+-
 |X| 

Exploring state (Player O, Depth 0):
X|O| 
-+-+-
 | | 
-+-+-
 | |X

Exploring state (Player X, Depth 1):
X| |O
-+-+-
 | | 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|X|O
-+-+-
 | | 
-+-+-
 | | 

Exploring state (Player X, Depth 1):
X| | 
-+-+-
O| | 
-+-+-
 | | 

Exploring state (Player O, Depth 0):
X|X| 
-+-+-
O| | 
-+-+-
 | | 

Exploring state (Player X, Depth 1):
X| | 
-+-+-
 |O| 
-+-+-
 | | 

Explor